**Summary:** LangGraph pipeline (plan → search → synthesize → Telegram) via Gradio with live step progress, run button disabled while busy, session checkpointer, and Telegram plain text (markdown stripped).

# Opportunities researcher


In [ ]:
import os
import re
import uuid
from datetime import date, timedelta
from typing import Annotated, TypedDict

import httpx
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.messages import AIMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

load_dotenv(override=True)

In [ ]:
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
MODEL = "openai/gpt-4o-mini"
TELEGRAM_TIMEOUT = 25.0
TELEGRAM_MAX_LENGTH = 4096
SERPER_TOP = 5
MAX_FETCH_PAGES = 12
MAX_PAGE_CHARS = 1800
HTTP_FETCH_TIMEOUT = 30.0
OPPORTUNITY_WINDOW_DAYS = 30

DEFAULT_RESEARCH_BRIEF = (
    "Find accelerators, hackathons, fellowships, grants, and incubators relevant to this profile. "
    "Prioritize items with application deadlines, event dates, or cohort starts inside the stated 30-day window."
)


def opportunity_window_banner() -> str:
    window_start_date = date.today()
    window_end_date = window_start_date + timedelta(days=OPPORTUNITY_WINDOW_DAYS - 1)
    return (
        f"Date window (inclusive, {OPPORTUNITY_WINDOW_DAYS} calendar days): "
        f"{window_start_date.isoformat()} through {window_end_date.isoformat()}. "
        "Do not include deadlines or events outside this range."
    )


def _require_env(name: str) -> str | None:
    value = (os.getenv(name) or "").strip()
    return value or None


google_serper_search = None
if _require_env("SERPER_API_KEY"):
    try:
        google_serper_search = GoogleSerperAPIWrapper(k=SERPER_TOP)
    except Exception:
        google_serper_search = None

openrouter_chat_model = ChatOpenAI(
    model=MODEL,
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=OPENROUTER_BASE,
    temperature=0.35,
)

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID")


def send_telegram(message_text: str) -> None:
    if not TELEGRAM_BOT_TOKEN or not TELEGRAM_CHAT_ID:
        return
    if len(message_text) > TELEGRAM_MAX_LENGTH:
        message_text = message_text[: TELEGRAM_MAX_LENGTH - 4] + "..."
    telegram_api_url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    with httpx.Client(timeout=TELEGRAM_TIMEOUT) as http_client:
        http_response = http_client.post(
            telegram_api_url, json={"chat_id": TELEGRAM_CHAT_ID, "text": message_text}
        )
        http_response.raise_for_status()


def strip_markdown_for_telegram(text: str) -> str:
    """Flatten Markdown to plain text for Telegram sendMessage (no parse_mode)."""
    if not text:
        return ""
    t = text
    t = re.sub(r"\[([^\]]+)\]\((https?://[^)]+)\)", r"\1 — \2", t)
    t = re.sub(r"\[([^\]]+)\]\(([^)]+)\)", r"\1 — \2", t)
    t = re.sub(r"`+([^`]+)`+", r"\1", t)
    t = re.sub(r"~{1,2}([^~]+)~{1,2}", r"\1", t)
    t = re.sub(r"^#+\s*", "", t, flags=re.MULTILINE)
    for _ in range(10):
        prev = t
        t = re.sub(r"\*\*([^*]+)\*\*", r"\1", t)
        if t == prev:
            break
    t = re.sub(r"(?<!\*)\*(?!\*)([^*\n]+?)(?<!\*)\*(?!\*)", r"\1", t)
    for _ in range(10):
        prev = t
        t = re.sub(r"__([^_]+)__", r"\1", t)
        if t == prev:
            break
    t = re.sub(r"(?<!_)_(?!_)([^_\n]+?)(?<!_)_(?!_)", r"\1", t)
    t = t.replace("**", "").replace("__", "")
    return re.sub(r"\n{3,}", "\n\n", t).strip()


def profile_line(mode: str, name: str, url: str) -> str:
    name, url = name.strip(), url.strip()
    if mode == "individual":
        return f"Individual: {name} | LinkedIn: {url}"
    return f"Organization: {name} | Website: {url}"


def fetch_page_texts(urls: list[str]) -> list[str]:
    page_text_chunks: list[str] = []
    visited_page_urls: set[str] = set()
    request_headers = {
        "User-Agent": "Mozilla/5.0 (compatible; OpportunitiesResearcher/1.0; +https://example.local)"
    }
    with httpx.Client(
        timeout=HTTP_FETCH_TIMEOUT, follow_redirects=True, headers=request_headers
    ) as http_client:
        for page_url in urls:
            if not page_url or not page_url.startswith("http") or page_url in visited_page_urls:
                continue
            visited_page_urls.add(page_url)
            if len(page_text_chunks) >= MAX_FETCH_PAGES:
                break
            try:
                http_response = http_client.get(page_url)
                http_response.raise_for_status()
                soup = BeautifulSoup(http_response.text, "html.parser")
                for unwanted in soup(["script", "style", "noscript"]):
                    unwanted.decompose()
                raw_body_text = soup.get_text(separator=" ")
                raw_body_text = re.sub(r"\s+", " ", raw_body_text).strip()
                page_text_chunks.append(f"[{page_url}]\n{raw_body_text[:MAX_PAGE_CHARS]}")
            except Exception as fetch_error:
                page_text_chunks.append(f"[{page_url}]\n[fetch: {fetch_error}]\n")
    return page_text_chunks

In [ ]:
class GraphState(TypedDict):
    messages: Annotated[list, add_messages]
    mode: str
    profile_name: str
    profile_url: str
    user_query: str
    plan: str
    raw_notes: str
    report: str
    error: str

In [ ]:
def _prior_context(messages: list | None) -> str:
    if not messages or len(messages) < 2:
        return ""
    context_lines = []
    for prior_message in messages[:-1][-8:]:
        message_text_preview = (getattr(prior_message, "content", None) or "")[:500]
        if isinstance(prior_message, HumanMessage):
            context_lines.append(f"User: {message_text_preview}")
        elif isinstance(prior_message, AIMessage):
            context_lines.append(f"Assistant: {message_text_preview}")
    return "\n".join(context_lines)


def node_plan(state: GraphState) -> dict:
    profile_context_line = profile_line(state["mode"], state["profile_name"], state["profile_url"])
    user_request_text = state["user_query"].strip()
    prior_thread_context = _prior_context(state.get("messages"))
    planner_prompt = (
        f"Context: {profile_context_line}\n\n"
        f"Request:\n{user_request_text}\n\n"
        f"{opportunity_window_banner()} "
        "Each search query must help find programs whose application deadline, event date, or cohort start "
        "falls inside that window (not earlier, not later). Use concrete months/years in queries when useful.\n\n"
        "Output 5–7 lines. Each line is one Google-style query: accelerators, hackathons, fellowships, "
        "grants, or incubators for this profile. Bullets only, no intro."
    )
    if prior_thread_context:
        planner_prompt += f"\n\nEarlier in this thread:\n{prior_thread_context}\n"
    try:
        planner_llm_response = openrouter_chat_model.invoke([HumanMessage(content=planner_prompt)])
    except Exception as planner_error:
        return {"plan": "", "error": f"Planning (LLM) failed: {planner_error}"}
    plan_text = planner_llm_response.content if isinstance(planner_llm_response.content, str) else str(planner_llm_response.content)
    return {"plan": plan_text, "error": ""}


def node_research(state: GraphState) -> dict:
    if state.get("error"):
        return {"raw_notes": "", "error": state["error"]}
    if not google_serper_search:
        return {
            "raw_notes": "",
            "error": "Search unavailable: set SERPER_API_KEY in .env and restart the kernel.",
        }
    search_query_lines = [line.strip("- •\t ") for line in state["plan"].splitlines() if line.strip()]
    search_query_lines = search_query_lines[:8]
    if not search_query_lines:
        return {"raw_notes": "", "error": "No search queries were produced in the plan step."}
    serp_and_scrape_chunks: list[str] = []
    result_links_ordered: list[str] = []
    for search_query_line in search_query_lines:
        try:
            serper_json = google_serper_search.results(search_query_line)
            organic_results = serper_json.get("organic") or []
        except Exception as serper_error:
            serp_and_scrape_chunks.append(f"--- {search_query_line}\n[serper: {serper_error}]\n")
            continue
        for search_hit in organic_results[:SERPER_TOP]:
            hit_title = search_hit.get("title") or ""
            hit_link = search_hit.get("link") or ""
            snippet_text = search_hit.get("snippet") or ""
            serp_and_scrape_chunks.append(
                f"--- {search_query_line}\n{hit_title}\n{hit_link}\n{snippet_text}\n"
            )
            if hit_link:
                result_links_ordered.append(hit_link)
    deduplicated_urls: list[str] = []
    urls_already_added: set[str] = set()
    for result_url in result_links_ordered:
        if result_url not in urls_already_added:
            urls_already_added.add(result_url)
            deduplicated_urls.append(result_url)
    try:
        http_page_texts = fetch_page_texts(deduplicated_urls)
    except Exception as fetch_batch_error:
        http_page_texts = [f"[page fetch batch error: {fetch_batch_error}]"]
    if http_page_texts:
        serp_and_scrape_chunks.append("\n--- Page extracts ---\n" + "\n\n".join(http_page_texts))
    return {"raw_notes": "\n".join(serp_and_scrape_chunks), "error": ""}


def node_synthesize(state: GraphState) -> dict:
    if state.get("error"):
        return {"report": "", "error": state["error"]}
    profile_context_line = profile_line(state["mode"], state["profile_name"], state["profile_url"])
    prior_thread_context = _prior_context(state.get("messages"))
    synthesis_prompt = (
        f"Profile: {profile_context_line}\n\n"
        f"Request:\n{state['user_query']}\n\n"
        f"{opportunity_window_banner()}\n\n"
        f"Sources (search + page text):\n{state['raw_notes']}\n\n"
        "Write a report: ## Summary, ## Programs and events (within window only), ## Deadlines and links, ## Fit. "
        "List ONLY opportunities whose application deadline, hackathon date, fellowship start, or comparable "
        "date falls within the rolling window above. Exclude anything that is only later or only earlier. "
        "If sources lack dates, omit or say 'date unclear — excluded from shortlist'. Bullets; cite URLs; max ~1500 words."
    )
    if prior_thread_context:
        synthesis_prompt += f"\n\nEarlier in this thread:\n{prior_thread_context}\n"
    try:
        synthesis_response = openrouter_chat_model.invoke([HumanMessage(content=synthesis_prompt)])
    except Exception as synthesis_error:
        return {"report": "", "error": f"Report (LLM) failed: {synthesis_error}"}
    report_markdown = (
        synthesis_response.content if isinstance(synthesis_response.content, str) else str(synthesis_response.content)
    )
    return {"report": report_markdown, "messages": [AIMessage(content=report_markdown)], "error": ""}


def node_telegram(state: GraphState) -> dict:
    if state.get("error") or not (state.get("report") or "").strip():
        return {}
    telegram_digest_prompt = (
        f"{opportunity_window_banner()} "
        "Compress into one Telegram message: plain text only, no markdown (no **, no #, no [text](url) — write full URLs). "
        "Max 3200 chars. Only items inside that window. Lead with 2–3 lines, then bullets (names + URLs).\n\n"
        f"{state['report']}"
    )
    try:
        digest_response = openrouter_chat_model.invoke([HumanMessage(content=telegram_digest_prompt)])
        telegram_digest_text = (
            digest_response.content if isinstance(digest_response.content, str) else str(digest_response.content)
        )
        send_telegram(strip_markdown_for_telegram(telegram_digest_text))
    except Exception as telegram_error:
        return {"error": f"Telegram: {telegram_error}"}
    return {}

In [ ]:
opportunity_graph_builder = StateGraph(GraphState)
opportunity_graph_builder.add_node("plan", node_plan)
opportunity_graph_builder.add_node("research", node_research)
opportunity_graph_builder.add_node("synthesize", node_synthesize)
opportunity_graph_builder.add_node("telegram", node_telegram)
opportunity_graph_builder.add_edge(START, "plan")
opportunity_graph_builder.add_edge("plan", "research")
opportunity_graph_builder.add_edge("research", "synthesize")
opportunity_graph_builder.add_edge("synthesize", "telegram")
opportunity_graph_builder.add_edge("telegram", END)
graph_checkpoint_memory = MemorySaver()
compiled_opportunity_graph = opportunity_graph_builder.compile(checkpointer=graph_checkpoint_memory)

In [ ]:
display(Image(compiled_opportunity_graph.get_graph().draw_mermaid_png()))

In [ ]:
def _format_graph_output(final_graph_state: dict) -> str:
    error_message = (final_graph_state.get("error") or "").strip()
    report_body = (final_graph_state.get("report") or "").strip()
    if error_message and not report_body:
        return f"**Stopped:** {error_message}"
    if error_message and report_body:
        return f"{report_body}\n\n---\n_Note: {error_message}_"
    return report_body or "(No report returned.)"


def run_pipeline(thread_id: str, mode: str, profile_name: str, profile_url: str, user_query: str) -> str:
    if not _require_env("OPENROUTER_API_KEY"):
        return "**Configuration error:** set OPENROUTER_API_KEY in .env and restart the kernel."
    initial_graph_state: GraphState = {
        "messages": [HumanMessage(content=user_query)],
        "mode": mode if mode in ("individual", "business") else "individual",
        "profile_name": profile_name,
        "profile_url": profile_url,
        "user_query": user_query,
        "plan": "",
        "raw_notes": "",
        "report": "",
        "error": "",
    }
    checkpoint_config = {"configurable": {"thread_id": thread_id}}
    try:
        final_graph_state = compiled_opportunity_graph.invoke(initial_graph_state, config=checkpoint_config)
    except Exception as graph_error:
        return (
            f"**Run failed**\n\n{type(graph_error).__name__}: {graph_error}\n\n"
            "If this mentions subprocess or NotImplementedError on Windows, ensure you are not using Playwright "
            "in the notebook; this build uses HTTP fetch only."
        )
    return _format_graph_output(final_graph_state)

In [ ]:
import gradio as gr


def _progress_markdown(completed_steps: list[str], suffix: str = "") -> str:
    lines = ["### Progress", *[f"- ✓ **{step}**" for step in completed_steps]]
    if suffix:
        lines.append(suffix)
    return "\n".join(lines)


with gr.Blocks(title="Opportunities researcher") as gradio_app:
    gr.Markdown("Enter profile details and click **Run research**. Uses a fixed internal brief (30-day window).")
    profile_type_input = gr.Radio(["individual", "business"], value="individual", label="Profile type")
    profile_name_input = gr.Textbox(label="Name (person or business)", placeholder="Chimobi Ogudu")
    profile_url_input = gr.Textbox(
        label="LinkedIn or company website",
        placeholder="https://www.linkedin.com/in/... or https://...",
    )
    optional_focus = gr.Textbox(
        label="Optional focus (leave blank to use default brief)",
        placeholder="e.g. fintech in Africa — optional",
        lines=2,
    )
    session_thread_id_state = gr.State(value=None)
    run_button = gr.Button("Run research", variant="primary")
    progress_markdown_output = gr.Markdown(value="")
    report_markdown_output = gr.Markdown(label="Report")

    def run_with_optional_focus(profile_mode, name, url, focus_text, thread_id):
        brief = DEFAULT_RESEARCH_BRIEF
        if (focus_text or "").strip():
            brief = f"{DEFAULT_RESEARCH_BRIEF} Additional focus: {focus_text.strip()}"
        name_clean = (name or "").strip()
        url_clean = (url or "").strip()
        if not name_clean:
            yield (
                gr.update(interactive=True),
                "",
                "**Missing:** enter a name.",
                thread_id,
            )
            return
        if not url_clean:
            yield (
                gr.update(interactive=True),
                "",
                "**Missing:** enter a LinkedIn or website URL.",
                thread_id,
            )
            return
        if thread_id is None:
            thread_id = str(uuid.uuid4())

        yield gr.update(interactive=False), _progress_markdown([], "_Starting…_"), "", thread_id

        if not _require_env("OPENROUTER_API_KEY"):
            yield (
                gr.update(interactive=True),
                "",
                "**Configuration error:** set OPENROUTER_API_KEY in .env and restart the kernel.",
                thread_id,
            )
            return

        initial_graph_state: GraphState = {
            "messages": [HumanMessage(content=brief)],
            "mode": profile_mode if profile_mode in ("individual", "business") else "individual",
            "profile_name": name_clean,
            "profile_url": url_clean,
            "user_query": brief,
            "plan": "",
            "raw_notes": "",
            "report": "",
            "error": "",
        }
        checkpoint_config = {"configurable": {"thread_id": thread_id}}
        completed_steps: list[str] = []
        try:
            for update_chunk in compiled_opportunity_graph.stream(
                initial_graph_state, config=checkpoint_config, stream_mode="updates"
            ):
                for node_name in update_chunk.keys():
                    if node_name not in completed_steps:
                        completed_steps.append(node_name)
                yield (
                    gr.update(interactive=False),
                    _progress_markdown(completed_steps, "_Running…_"),
                    "",
                    thread_id,
                )
            snapshot = compiled_opportunity_graph.get_state(checkpoint_config)
            final_values = snapshot.values
            report = _format_graph_output(final_values)
            yield (
                gr.update(interactive=True),
                _progress_markdown(completed_steps, "_Done._"),
                report,
                thread_id,
            )
        except Exception as graph_error:
            yield (
                gr.update(interactive=True),
                _progress_markdown(completed_steps, f"_Stopped: {graph_error}_"),
                (
                    f"**Run failed**\n\n{type(graph_error).__name__}: {graph_error}\n\n"
                    "If this mentions subprocess or NotImplementedError on Windows, this build uses HTTP fetch only."
                ),
                thread_id,
            )

    run_button.click(
        fn=run_with_optional_focus,
        inputs=[profile_type_input, profile_name_input, profile_url_input, optional_focus, session_thread_id_state],
        outputs=[run_button, progress_markdown_output, report_markdown_output, session_thread_id_state],
    )

gradio_app.launch(share=False)